# Amazon Redshift: basic to intermediate SQL with `ecommdb`

This lesson uses a small e-commerce warehouse to teach Redshift directly: database and schema setup, table design, distribution, sort keys, analytical SQL, staging, `MERGE`, CTAS, temporary tables, query plans, monitoring, and maintenance.

> Run the SQL blocks in order in **Amazon Redshift Query Editor v2** or another SQL client connected to a **provisioned Redshift cluster**. The examples use only Redshift SQL; MySQL-specific features such as storage engines, indexes, and `AUTO_INCREMENT` are intentionally not used.

## 1. Create and connect to the database

Connect first to an existing database such as `dev`. `CREATE DATABASE` must run outside a transaction. Run it once, then create a new Query Editor connection to `ecommdb`.

```sql
CREATE DATABASE ecommdb;
```

After reconnecting to `ecommdb`, confirm the session.

```sql
SELECT current_database() AS database_name,
       current_user AS user_name,
       current_schema AS schema_name,
       GETDATE() AS cluster_timestamp;
```

## 2. How Redshift partitions data

Redshift does not use MySQL-style indexes or native table partitions. It organizes data through three separate ideas:

| Layer | Redshift feature | Purpose |
|---|---|---|
| Across compute slices | distribution style and distribution key | decides where rows live and whether joins require network movement |
| Inside each slice | sort key and 1 MB block zone maps | skips blocks and reduces scan work |
| External data in S3 | Spectrum partitions | prunes folders such as `order_year=2026/order_month=9` |

Distribution controls **placement**; a sort key controls **local ordering**. They solve different problems.

## 3. Create the teaching schema

The reset is limited to the `ecomm` schema. Leave the `DROP` statement commented unless a clean rerun is required.

```sql
-- DROP SCHEMA ecomm CASCADE;
CREATE SCHEMA IF NOT EXISTS ecomm;
SET search_path TO ecomm, public;
```

## 4. Dimension tables: `DISTSTYLE ALL`

Small, frequently joined dimensions are copied to every compute node with `DISTSTYLE ALL`. This removes redistribution during joins, at the cost of extra storage and load work. Primary and foreign keys in Redshift are **informational, not enforced**; declare them only when the data is genuinely valid.

```sql
CREATE TABLE IF NOT EXISTS ecomm.customers (
    customer_id       BIGINT       NOT NULL,
    customer_name     VARCHAR(100) NOT NULL,
    email             VARCHAR(200),
    city              VARCHAR(80),
    state_code        CHAR(2),
    customer_segment  VARCHAR(20),
    signup_date       DATE         NOT NULL,
    PRIMARY KEY (customer_id)
)
DISTSTYLE ALL
SORTKEY (customer_id);

CREATE TABLE IF NOT EXISTS ecomm.products (
    product_id       BIGINT        NOT NULL,
    product_name     VARCHAR(150)  NOT NULL,
    category         VARCHAR(60)   NOT NULL,
    unit_price       DECIMAL(12,2) NOT NULL,
    active_flag      BOOLEAN       DEFAULT TRUE,
    PRIMARY KEY (product_id)
)
DISTSTYLE ALL
SORTKEY (product_id);
```

## 5. Fact tables: co-locate the order family

`orders`, `order_items`, and `invoices` use the same `DISTKEY(order_id)`. Rows for the same order hash to the same slice, so their largest joins can be local. The leading sort column is the date commonly used by range filters.

```sql
CREATE TABLE IF NOT EXISTS ecomm.orders (
    order_id          BIGINT        NOT NULL,
    customer_id       BIGINT        NOT NULL,
    order_ts          TIMESTAMP     NOT NULL,
    order_date        DATE          NOT NULL,
    order_status      VARCHAR(20)   NOT NULL,
    shipping_amount   DECIMAL(12,2) DEFAULT 0,
    PRIMARY KEY (order_id),
    FOREIGN KEY (customer_id) REFERENCES ecomm.customers(customer_id)
)
DISTSTYLE KEY
DISTKEY (order_id)
COMPOUND SORTKEY (order_date, order_id);

CREATE TABLE IF NOT EXISTS ecomm.order_items (
    order_id          BIGINT        NOT NULL,
    line_number       SMALLINT      NOT NULL,
    product_id        BIGINT        NOT NULL,
    quantity          INTEGER       NOT NULL,
    unit_price        DECIMAL(12,2) NOT NULL,
    discount_amount   DECIMAL(12,2) DEFAULT 0,
    PRIMARY KEY (order_id, line_number),
    FOREIGN KEY (order_id) REFERENCES ecomm.orders(order_id),
    FOREIGN KEY (product_id) REFERENCES ecomm.products(product_id)
)
DISTSTYLE KEY
DISTKEY (order_id)
COMPOUND SORTKEY (order_id, product_id);

CREATE TABLE IF NOT EXISTS ecomm.invoices (
    invoice_id       BIGINT        NOT NULL,
    order_id         BIGINT        NOT NULL,
    invoice_date     DATE          NOT NULL,
    invoice_status   VARCHAR(20)   NOT NULL,
    invoice_amount   DECIMAL(14,2) NOT NULL,
    paid_at          TIMESTAMP,
    PRIMARY KEY (invoice_id),
    FOREIGN KEY (order_id) REFERENCES ecomm.orders(order_id)
)
DISTSTYLE KEY
DISTKEY (order_id)
COMPOUND SORTKEY (invoice_date, order_id);
```

## 6. An independent purchase fact

A purchase represents stock bought from a supplier, not a customer order. `DISTSTYLE AUTO` lets Redshift choose and later adjust distribution. AUTO is a strong default when the dominant join pattern is not yet known.

```sql
CREATE TABLE IF NOT EXISTS ecomm.purchases (
    purchase_id      BIGINT        NOT NULL,
    supplier_id      BIGINT        NOT NULL,
    product_id       BIGINT        NOT NULL,
    purchase_date    DATE          NOT NULL,
    quantity         INTEGER       NOT NULL,
    unit_cost        DECIMAL(12,2) NOT NULL,
    received_date    DATE,
    PRIMARY KEY (purchase_id)
)
DISTSTYLE AUTO
SORTKEY AUTO;
```

## 7. Load a small repeatable dataset

`TRUNCATE` makes this classroom seed repeatable. Production-scale Redshift loads should use parallel `COPY` from S3 rather than many row-by-row inserts.

```sql
TRUNCATE TABLE ecomm.invoices;
TRUNCATE TABLE ecomm.order_items;
TRUNCATE TABLE ecomm.orders;
TRUNCATE TABLE ecomm.purchases;
TRUNCATE TABLE ecomm.products;
TRUNCATE TABLE ecomm.customers;

INSERT INTO ecomm.customers VALUES
(1, 'Asha Rao',   'asha@example.com',  'Bengaluru', 'KA', 'consumer',  '2025-01-10'),
(2, 'Bilal Khan', 'bilal@example.com', 'Hyderabad', 'TS', 'corporate', '2025-02-05'),
(3, 'Charu Sen',  'charu@example.com', 'Kolkata',   'WB', 'consumer',  '2025-03-18'),
(4, 'Deepak Iyer','deepak@example.com','Chennai',   'TN', 'small_biz', '2025-04-02'),
(5, 'Eva Thomas', 'eva@example.com',   'Kochi',     'KL', 'consumer',  '2025-05-14');

INSERT INTO ecomm.products VALUES
(101, 'Mechanical Keyboard', 'electronics', 4500.00, TRUE),
(102, 'USB-C Hub',           'electronics', 2200.00, TRUE),
(103, 'Desk Chair',          'furniture',   8500.00, TRUE),
(104, 'Notebook Pack',       'stationery',   350.00, TRUE),
(105, 'Monitor Arm',         'furniture',   3200.00, TRUE);

INSERT INTO ecomm.orders VALUES
(1001, 1, '2026-07-02 10:15:00', '2026-07-02', 'delivered', 120.00),
(1002, 2, '2026-07-15 14:30:00', '2026-07-15', 'delivered',   0.00),
(1003, 1, '2026-08-01 09:00:00', '2026-08-01', 'shipped',   90.00),
(1004, 3, '2026-08-09 18:20:00', '2026-08-09', 'delivered', 60.00),
(1005, 4, '2026-08-22 11:10:00', '2026-08-22', 'cancelled',  0.00),
(1006, 5, '2026-09-01 16:45:00', '2026-09-01', 'placed',   100.00);

INSERT INTO ecomm.order_items VALUES
(1001, 1, 101, 1, 4500.00, 200.00),
(1001, 2, 104, 2,  350.00,   0.00),
(1002, 1, 103, 1, 8500.00, 500.00),
(1003, 1, 102, 2, 2200.00, 100.00),
(1004, 1, 104, 5,  350.00,  50.00),
(1004, 2, 105, 1, 3200.00,   0.00),
(1005, 1, 101, 1, 4500.00,   0.00),
(1006, 1, 105, 2, 3200.00, 250.00);

INSERT INTO ecomm.invoices VALUES
(5001, 1001, '2026-07-02', 'paid',    5120.00, '2026-07-02 10:18:00'),
(5002, 1002, '2026-07-15', 'paid',    8000.00, '2026-07-15 14:35:00'),
(5003, 1003, '2026-08-01', 'pending', 4390.00, NULL),
(5004, 1004, '2026-08-09', 'paid',    4960.00, '2026-08-09 18:23:00'),
(5005, 1006, '2026-09-01', 'pending', 6250.00, NULL);

INSERT INTO ecomm.purchases VALUES
(9001, 701, 101, '2026-06-15', 20, 3100.00, '2026-06-20'),
(9002, 702, 104, '2026-07-01', 200, 180.00, '2026-07-05'),
(9003, 701, 105, '2026-07-20', 40, 2100.00, '2026-07-27'),
(9004, 703, 102, '2026-08-10', 60, 1450.00, NULL);
```

## 8. Validate before trusting constraints

Because key constraints are not enforced, test uniqueness and relationships after every load. Each result should be zero.

```sql
SELECT 'duplicate customers' AS check_name, COUNT(*) - COUNT(DISTINCT customer_id) AS violations
FROM ecomm.customers
UNION ALL
SELECT 'duplicate orders', COUNT(*) - COUNT(DISTINCT order_id)
FROM ecomm.orders
UNION ALL
SELECT 'orphan orders', COUNT(*)
FROM ecomm.orders o LEFT JOIN ecomm.customers c ON c.customer_id = o.customer_id
WHERE c.customer_id IS NULL
UNION ALL
SELECT 'orphan order items', COUNT(*)
FROM ecomm.order_items i LEFT JOIN ecomm.orders o ON o.order_id = i.order_id
WHERE o.order_id IS NULL;
```

## 9. Basic analytical queries

Calculate line revenue once and aggregate at the requested grain. `DECIMAL` keeps currency arithmetic exact.

```sql
SELECT
    o.order_id,
    c.customer_name,
    o.order_date,
    o.order_status,
    SUM(i.quantity * i.unit_price - i.discount_amount) AS merchandise_amount,
    MAX(o.shipping_amount) AS shipping_amount,
    SUM(i.quantity * i.unit_price - i.discount_amount) + MAX(o.shipping_amount) AS order_total
FROM ecomm.orders o
JOIN ecomm.customers c ON c.customer_id = o.customer_id
JOIN ecomm.order_items i ON i.order_id = o.order_id
WHERE o.order_status <> 'cancelled'
GROUP BY o.order_id, c.customer_name, o.order_date, o.order_status
ORDER BY o.order_date, o.order_id;
```

```sql
SELECT
    DATE_TRUNC('month', o.order_date)::DATE AS order_month,
    p.category,
    COUNT(DISTINCT o.order_id) AS orders,
    SUM(i.quantity) AS units,
    SUM(i.quantity * i.unit_price - i.discount_amount) AS net_revenue
FROM ecomm.orders o
JOIN ecomm.order_items i ON i.order_id = o.order_id
JOIN ecomm.products p ON p.product_id = i.product_id
WHERE o.order_status <> 'cancelled'
GROUP BY 1, 2
ORDER BY 1, net_revenue DESC;
```

## 10. Conditional aggregation and null handling

```sql
SELECT
    c.state_code,
    COUNT(DISTINCT o.order_id) AS all_orders,
    COUNT(DISTINCT CASE WHEN o.order_status = 'delivered' THEN o.order_id END) AS delivered_orders,
    COALESCE(SUM(CASE WHEN o.order_status = 'delivered'
                      THEN i.quantity * i.unit_price - i.discount_amount END), 0) AS delivered_revenue
FROM ecomm.customers c
LEFT JOIN ecomm.orders o ON o.customer_id = c.customer_id
LEFT JOIN ecomm.order_items i ON i.order_id = o.order_id
GROUP BY c.state_code
ORDER BY delivered_revenue DESC;
```

## 11. Window functions

The inner query first produces one row per order. The outer query computes customer sequence and running spend without collapsing rows. Include a deterministic tie-breaker in the window order.

```sql
WITH order_totals AS (
    SELECT
        o.order_id, o.customer_id, o.order_ts,
        SUM(i.quantity * i.unit_price - i.discount_amount) + MAX(o.shipping_amount) AS order_total
    FROM ecomm.orders o
    JOIN ecomm.order_items i ON i.order_id = o.order_id
    WHERE o.order_status <> 'cancelled'
    GROUP BY o.order_id, o.customer_id, o.order_ts
)
SELECT
    order_id, customer_id, order_ts, order_total,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_ts, order_id) AS customer_order_number,
    SUM(order_total) OVER (
        PARTITION BY customer_id
        ORDER BY order_ts, order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_customer_spend
FROM order_totals
ORDER BY customer_id, order_ts, order_id;
```

## 12. Avoid a many-to-many join error

Joining multiple one-to-many child tables directly multiplies rows. Aggregate each child to one row per order before joining.

```sql
WITH item_totals AS (
    SELECT order_id,
           SUM(quantity * unit_price - discount_amount) AS merchandise_amount
    FROM ecomm.order_items
    GROUP BY order_id
),
invoice_totals AS (
    SELECT order_id,
           SUM(invoice_amount) AS invoiced_amount,
           SUM(CASE WHEN invoice_status = 'paid' THEN invoice_amount ELSE 0 END) AS paid_amount
    FROM ecomm.invoices
    GROUP BY order_id
)
SELECT o.order_id, i.merchandise_amount,
       COALESCE(v.invoiced_amount, 0) AS invoiced_amount,
       COALESCE(v.paid_amount, 0) AS paid_amount
FROM ecomm.orders o
JOIN item_totals i ON i.order_id = o.order_id
LEFT JOIN invoice_totals v ON v.order_id = o.order_id
ORDER BY o.order_id;
```

## 13. Temporary tables for intermediate results

A temporary table exists only for the current session. Choose its distribution to match the next large join. `ANALYZE` gives the optimizer statistics for better plans.

```sql
DROP TABLE IF EXISTS temp_recent_order_totals;

CREATE TEMP TABLE temp_recent_order_totals
DISTSTYLE KEY
DISTKEY (order_id)
SORTKEY (order_date)
AS
SELECT
    o.order_id, o.customer_id, o.order_date,
    SUM(i.quantity * i.unit_price - i.discount_amount) AS merchandise_amount
FROM ecomm.orders o
JOIN ecomm.order_items i ON i.order_id = o.order_id
WHERE o.order_date >= DATEADD(month, -3, CURRENT_DATE)
  AND o.order_status <> 'cancelled'
GROUP BY o.order_id, o.customer_id, o.order_date;

ANALYZE temp_recent_order_totals;
SELECT * FROM temp_recent_order_totals ORDER BY order_date, order_id;
```

## 14. CTAS for a reusable aggregate

Create Table As Select is efficient because Redshift writes query output directly into columnar blocks. This rebuild pattern is useful for derived tables.

```sql
DROP TABLE IF EXISTS ecomm.monthly_category_sales;

CREATE TABLE ecomm.monthly_category_sales
DISTSTYLE AUTO
SORTKEY (order_month, category)
AS
SELECT
    DATE_TRUNC('month', o.order_date)::DATE AS order_month,
    p.category,
    COUNT(DISTINCT o.order_id) AS order_count,
    SUM(i.quantity) AS unit_count,
    SUM(i.quantity * i.unit_price - i.discount_amount) AS net_revenue
FROM ecomm.orders o
JOIN ecomm.order_items i ON i.order_id = o.order_id
JOIN ecomm.products p ON p.product_id = i.product_id
WHERE o.order_status <> 'cancelled'
GROUP BY 1, 2;

ANALYZE ecomm.monthly_category_sales;
SELECT * FROM ecomm.monthly_category_sales ORDER BY order_month, net_revenue DESC;
```

## 15. Stage and upsert with `MERGE`

Load a batch into a staging table, validate it, then merge it atomically. The source must not contain multiple rows matching the same target row.

```sql
DROP TABLE IF EXISTS stage_products;
CREATE TEMP TABLE stage_products (LIKE ecomm.products);

INSERT INTO stage_products VALUES
(102, 'USB-C Hub - 8 Port', 'electronics', 2400.00, TRUE),
(106, 'Standing Desk',      'furniture',  14500.00, TRUE);

MERGE INTO ecomm.products AS target
USING stage_products AS source
ON target.product_id = source.product_id
WHEN MATCHED THEN UPDATE SET
    product_name = source.product_name,
    category = source.category,
    unit_price = source.unit_price,
    active_flag = source.active_flag
WHEN NOT MATCHED THEN INSERT
    (product_id, product_name, category, unit_price, active_flag)
VALUES
    (source.product_id, source.product_name, source.category, source.unit_price, source.active_flag);

SELECT * FROM ecomm.products WHERE product_id IN (102, 106) ORDER BY product_id;
```

## 16. Transactions

Use a transaction when related changes must succeed or fail together. Redshift is optimized for set-based batches, not high-frequency single-row OLTP.

```sql
BEGIN;

UPDATE ecomm.orders
SET order_status = 'shipped'
WHERE order_id = 1006
  AND order_status = 'placed';

UPDATE ecomm.invoices
SET invoice_status = 'paid', paid_at = GETDATE()
WHERE order_id = 1006
  AND invoice_status = 'pending';

COMMIT;
```

Use `ROLLBACK` instead of `COMMIT` when validation fails.

## 17. Bulk loading from S3

Replace the URI and region. `IAM_ROLE default` uses the IAM role associated with the provisioned cluster. A prefix or manifest containing multiple similarly sized files allows slices to load in parallel. The column list protects against accidental source-column reordering.

```sql
COPY ecomm.orders (
    order_id, customer_id, order_ts, order_date, order_status, shipping_amount
)
FROM 's3://your-bucket/ecomm/orders/'
IAM_ROLE default
REGION 'ap-south-1'
FORMAT AS CSV
IGNOREHEADER 1
DATEFORMAT 'auto'
TIMEFORMAT 'auto'
EMPTYASNULL
BLANKSASNULL
COMPUPDATE ON
STATUPDATE ON;
```

Validate a file without loading rows by adding `NOLOAD` to the `COPY` command. Inspect failures with:

```sql
SELECT query_id, start_time, TRIM(file_name) AS file_name, line_number,
       TRIM(column_name) AS column_name, TRIM(error_message) AS error_message
FROM sys_load_error_detail
ORDER BY start_time DESC
LIMIT 20;
```

## 18. Inspect physical table design

`SVV_TABLE_INFO` shows table size, skew, sort health, and statistics health. Very high `skew_rows` can indicate a poor distribution key; a high `unsorted` percentage can reduce zone-map effectiveness. Some metrics are null for very small tables.

```sql
SELECT
    "schema", "table", diststyle, sortkey1,
    size AS size_mb, tbl_rows, skew_rows, unsorted, stats_off
FROM svv_table_info
WHERE "schema" = 'ecomm'
ORDER BY size DESC, "table";
```

Use `PG_TABLE_DEF` to confirm encoded columns, distribution keys, and sort-key positions. `PG_TABLE_DEF` follows the current `search_path`, so include `ecomm` first.

```sql
SET search_path TO ecomm, public;
SELECT tablename, "column", type, encoding, distkey, sortkey
FROM pg_table_def
WHERE schemaname = 'ecomm'
ORDER BY tablename, sortkey, "column";
```

## 19. Read an execution plan

`EXPLAIN` does not run the query. Look for scan filters, join types, row estimates, and data movement. `DS_DIST_NONE` means joined rows are already co-located; `DS_DIST_ALL_NONE` commonly appears when the inner table uses `DISTSTYLE ALL`. Redistribution labels such as `DS_DIST_BOTH` deserve investigation on large joins.

```sql
EXPLAIN
SELECT o.order_date, SUM(i.quantity * i.unit_price - i.discount_amount) AS revenue
FROM ecomm.orders o
JOIN ecomm.order_items i ON i.order_id = o.order_id
WHERE o.order_date BETWEEN DATE '2026-08-01' AND DATE '2026-08-31'
GROUP BY o.order_date
ORDER BY o.order_date;
```

## 20. Sort-key filtering and zone maps

Filter the stored sort column directly. Wrapping it in a function can make block pruning harder. Use a half-open range for timestamps.

```sql
-- Good: direct range predicate on the leading sort key.
SELECT order_id, customer_id, order_date, order_status
FROM ecomm.orders
WHERE order_date >= DATE '2026-08-01'
  AND order_date <  DATE '2026-09-01'
ORDER BY order_date, order_id;

-- For a TIMESTAMP sort column, use the same half-open pattern.
SELECT order_id, order_ts
FROM ecomm.orders
WHERE order_ts >= TIMESTAMP '2026-08-01 00:00:00'
  AND order_ts <  TIMESTAMP '2026-09-01 00:00:00';
```

## 21. Maintenance: statistics and sort order

Automatic Table Optimization and automatic analyze/vacuum handle much routine work. After an unusually large load or heavy update/delete cycle, inspect table health before running manual maintenance. `VACUUM` and `ANALYZE` are separate commands and should not be wrapped in an explicit transaction.

```sql
ANALYZE ecomm.orders;
ANALYZE ecomm.order_items;

-- Run only when svv_table_info shows that sorting is needed.
VACUUM SORT ONLY ecomm.orders;
VACUUM SORT ONLY ecomm.order_items;
```

## 22. Monitor recent queries on a provisioned cluster

The `SYS` views provide a consistent monitoring interface. Duration columns are in microseconds, so divide by one million for seconds. Visibility depends on user permissions.

```sql
SELECT
    query_id, status, start_time, end_time,
    ROUND(elapsed_time / 1000000.0, 3) AS elapsed_seconds,
    ROUND(queue_time / 1000000.0, 3) AS queue_seconds,
    ROUND(execution_time / 1000000.0, 3) AS execution_seconds,
    LEFT(query_text, 200) AS query_text
FROM sys_query_history
WHERE start_time >= DATEADD(hour, -1, GETDATE())
ORDER BY start_time DESC
LIMIT 25;
```

## 23. Spectrum partitions: a different kind of partitioning

External tables over S3 can use folder partitions. This is independent of Redshift distribution and sort keys. The external schema, Glue Data Catalog database, IAM permissions, and S3 data must already exist before this template can run. Partition columns are declared after the ordinary columns and are not stored inside each Parquet row.

```sql
-- Example external table; replace the schema and S3 locations.
CREATE EXTERNAL TABLE spectrum.order_events (
    order_id      BIGINT,
    customer_id   BIGINT,
    event_ts      TIMESTAMP,
    event_name    VARCHAR(40)
)
PARTITIONED BY (event_year INTEGER, event_month INTEGER)
STORED AS PARQUET
LOCATION 's3://your-bucket/ecomm/order_events/';

ALTER TABLE spectrum.order_events
ADD PARTITION (event_year=2026, event_month=9)
LOCATION 's3://your-bucket/ecomm/order_events/event_year=2026/event_month=9/';

-- Both partition predicates are important for S3 partition pruning.
SELECT event_name, COUNT(*)
FROM spectrum.order_events
WHERE event_year = 2026 AND event_month = 9
GROUP BY event_name;
```

## 24. Distribution decision guide

| Choice | Use it when | Watch for |
|---|---|---|
| `AUTO` | starting point; workload is evolving | inspect Redshift's eventual choice |
| `EVEN` | no stable join key; rows should spread uniformly | large joins may redistribute |
| `KEY` | two large tables repeatedly join on the same high-cardinality key | nulls or hot values can create skew |
| `ALL` | small, slowly changing table joins frequently to large facts | copy exists on every node; avoid for large tables |

A good distribution key has enough distinct values, distributes rows evenly, and matches an important large-table join. A column used only in filters is usually a sort-key candidate, not automatically a distribution-key candidate.

## 25. Sort-key decision guide

A compound sort key gives priority to its leading column and suits predictable range filters such as dates. Interleaved sort keys give equal weight to several filter columns, but carry higher maintenance cost and are rarely the first choice for append-heavy facts. `SORTKEY AUTO` allows Redshift to manage the decision.

Practical rules:

1. Put the most common selective range-filter column first.
2. Keep the key short; every added column has a cost.
3. Load roughly in sort-key order when practical.
4. Measure scanned blocks and elapsed time on representative data; a tiny seed dataset cannot demonstrate MPP performance.

## 26. End-to-end practice query

Find each month's top category by revenue and its share of that month's revenue.

```sql
WITH category_sales AS (
    SELECT
        DATE_TRUNC('month', o.order_date)::DATE AS order_month,
        p.category,
        SUM(i.quantity * i.unit_price - i.discount_amount) AS revenue
    FROM ecomm.orders o
    JOIN ecomm.order_items i ON i.order_id = o.order_id
    JOIN ecomm.products p ON p.product_id = i.product_id
    WHERE o.order_status <> 'cancelled'
    GROUP BY 1, 2
),
ranked AS (
    SELECT
        order_month, category, revenue,
        SUM(revenue) OVER (PARTITION BY order_month) AS month_revenue,
        ROW_NUMBER() OVER (PARTITION BY order_month ORDER BY revenue DESC, category) AS revenue_rank
    FROM category_sales
)
SELECT
    order_month, category, revenue,
    ROUND(100.0 * revenue / NULLIF(month_revenue, 0), 2) AS month_revenue_pct
FROM ranked
WHERE revenue_rank = 1
ORDER BY order_month;
```

## 27. Cleanup

Cleanup is intentionally not automatic. This removes only the teaching schema inside `ecommdb`.

```sql
-- DROP SCHEMA ecomm CASCADE;
```

To remove the database, disconnect all sessions from `ecommdb`, connect to another database such as `dev`, and run:

```sql
-- DROP DATABASE ecommdb;
```

## 28. Key takeaways

- Redshift is a columnar MPP analytical database; write set-based SQL.
- Distribution reduces network movement; sort keys enable block pruning.
- Co-locate large fact tables that repeatedly join on the same key.
- Treat primary and foreign keys as optimizer promises and validate them yourself.
- Prefer parallel `COPY` for bulk ingestion, staging plus `MERGE` for upserts, and CTAS for derived tables.
- Use `EXPLAIN`, `SVV_TABLE_INFO`, `PG_TABLE_DEF`, and `SYS_QUERY_HISTORY` to verify design decisions.